# Benchmark: Modelos de Reconocimiento LSM

Compara los tres modelos sobre el mismo split de test:

| # | Modelo | Tipo |
|---|--------|------|
| 1 | **DistanceModel** (kNN) | Nube de puntos — distancia |
| 2 | **PointNetModel** | Nube de puntos — probabilístico |
| 3 | **LSMTransformer** | Transformer con atención temporal |

**Requisitos antes de correr:**
```
pip install torch pandas pyarrow scikit-learn scipy numpy matplotlib seaborn
```

**Estructura esperada:**
```
proyecto/
  src/
    dataset.py
    augmentations.py
    model.py
  pointcloud_models.py
  corpus_LSM_esp/
    lsm_dataset.parquet
  runs/
    exp01/
      best_model.pt     ← generado por src/train.py
  notebooks/
    benchmark.ipynb     ← este archivo
```

## 0. Configuración y paths

In [2]:
import sys
import os

# Ajustar paths para imports desde notebooks/
PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

# ── Configuración ──────────────────────────────────────────────────────────
PARQUET_PATH      = "../corpus_LSM_esp/lsm_dataset.parquet"
TRANSFORMER_CKPT  = "../runs/exp03/best_model.pt"   # checkpoint de LSMTransformer
SEED              = 42
SCORE_THRESH      = 0.3

# Hiperparámetros PointNet (deben coincidir con los usados en entrenamiento)
POINTNET_EPOCHS   = 50
POINTNET_LR       = 1e-3
POINTNET_BATCH    = 32

# kNN
KNN_K = 5

print("Paths configurados.")
print(f"  Parquet : {PARQUET_PATH}")
print(f"  Ckpt    : {TRANSFORMER_CKPT}")

Paths configurados.
  Parquet : ../corpus_LSM_esp/lsm_dataset.parquet
  Ckpt    : ../runs/exp03/best_model.pt


## 1. Carga del dataset y split

In [3]:
import numpy as np
import torch
from torch.utils.data import DataLoader

from dataset import split_dataset, collate_fn, TMAX, INPUT_DIM, N_LANDMARKS

# Split idéntico al usado en entrenamiento (misma semilla)
train_ds, val_ds, test_ds = split_dataset(
    PARQUET_PATH,
    train_ratio=0.70,
    val_ratio=0.15,
    seed=SEED,
    augment_train=None,   # sin aumentación para benchmark
    score_thresh=SCORE_THRESH,
)

NUM_CLASSES = train_ds.num_classes
IDX2GLOSA   = train_ds.idx2glosa

print(f"Clases: {NUM_CLASSES}")
print(f"Train : {len(train_ds)} muestras")
print(f"Val   : {len(val_ds)}   muestras")
print(f"Test  : {len(test_ds)}  muestras")

[split_dataset] Cargando ../corpus_LSM_esp/lsm_dataset.parquet …
[split_dataset] 2447 videos | 249 glosas únicas
[split_dataset] train=1718 | val=402 | test=327
Clases: 249
Train : 1718 muestras
Val   : 402   muestras
Test  : 327  muestras


In [4]:
def dataset_to_numpy(ds):
    """
    Convierte un LSMDataset a arrays numpy.
    Los items tienen keypoints aplanados (T, 266) con longitud variable.
    Se hace padding a TMAX para poder apilarlos en un array uniforme.

    Returns
    -------
    X : (N, TMAX, 133, 2)  float32
    y : (N,)               int64
    """
    X_list, y_list = [], []

    for item in ds:
        T    = item["T"]
        kpts = item["keypoints"][:T].numpy()        # (T, 266)
        kpts_3d = kpts.reshape(T, N_LANDMARKS, 2)   # (T, 133, 2)

        # Padding a TMAX
        if T < TMAX:
            pad = np.zeros((TMAX - T, N_LANDMARKS, 2), dtype=np.float32)
            kpts_3d = np.concatenate([kpts_3d, pad], axis=0)

        X_list.append(kpts_3d)
        y_list.append(item["label"].item())

    return np.stack(X_list).astype(np.float32), np.array(y_list, dtype=np.int64)


print("Convirtiendo splits a numpy...")
X_train, y_train = dataset_to_numpy(train_ds)
X_val,   y_val   = dataset_to_numpy(val_ds)
X_test,  y_test  = dataset_to_numpy(test_ds)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")

Convirtiendo splits a numpy...
X_train: (1718, 200, 133, 2)  y_train: (1718,)
X_test : (327, 200, 133, 2)   y_test : (327,)


---
## 2. Modelo 1 — DistanceModel (kNN)

In [6]:
from pointcloud_models import DistanceModel

dist_model = DistanceModel(k=KNN_K, metric="euclidean")
dist_model.fit(X_train, y_train)

print("DistanceModel entrenado.")

[DistanceModel] Extrayendo descriptores de 1718 muestras...
[DistanceModel] Descriptor dim: 63
[DistanceModel] Entrenado. k=5, clases=249
DistanceModel entrenado.


In [7]:
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score, classification_report

dist_preds  = dist_model.predict(X_test)
dist_probas = dist_model.predict_proba(X_test)

dist_top1 = accuracy_score(y_test, dist_preds)
dist_top3 = top_k_accuracy_score(y_test, dist_probas, k=3, labels=list(range(NUM_CLASSES)))
dist_f1   = f1_score(y_test, dist_preds, average="macro",
                     labels=list(range(NUM_CLASSES)), zero_division=0)

print("=" * 40)
print("DistanceModel (kNN) — Resultados Test")
print("=" * 40)
print(f"  Top-1 Accuracy : {dist_top1:.4f}")
print(f"  Top-3 Accuracy : {dist_top3:.4f}")
print(f"  Macro F1       : {dist_f1:.4f}")

DistanceModel (kNN) — Resultados Test
  Top-1 Accuracy : 0.0367
  Top-3 Accuracy : 0.0795
  Macro F1       : 0.0264


In [8]:
# Predicción individual de ejemplo
sample_idx = 0
sample_seq = X_test[sample_idx]   # (TMAX, 133, 2)
true_label = IDX2GLOSA[y_test[sample_idx]]

result = dist_model.predict_single(sample_seq, top_k=5)

print(f"Etiqueta real : {true_label}")
print(f"Predicción    : {IDX2GLOSA[result['predicted_class']]}  "
      f"(conf={result['confidence']:.3f})")
print("Top-5:")
for r in result["top_k"]:
    print(f"  {IDX2GLOSA[r['class']]:<20} {r['prob']:.3f}")

Etiqueta real : 001
Predicción    : 046  (conf=0.215)
Top-5:
  046                  0.215
  065                  0.208
  051                  0.195
  008                  0.192
  002                  0.190


---
## 3. Modelo 2 — PointNetModel (probabilístico)

In [9]:
from pointcloud_models import PointNetTrainer

pn_trainer = PointNetTrainer(
    num_classes=NUM_CLASSES,
    latent_dim=256,
    dropout=0.3,
    lr=POINTNET_LR,
    epochs=POINTNET_EPOCHS,
    batch_size=POINTNET_BATCH,
    patience=10,
    temperature=1.0,
    tmax=TMAX,
)

pn_trainer.fit(X_train, y_train)
print("PointNetModel entrenado.")

[PointNetTrainer] Preparando datos (1718 muestras)...
[PointNetTrainer] Entrenando en cuda...
 Epoch |     Loss |    Acc
------------------------------
     1 |   5.5280 |  0.003
     5 |   4.7111 |  0.041
    10 |   3.9103 |  0.159
    15 |   3.3758 |  0.271
    20 |   3.0364 |  0.367
    25 |   2.7843 |  0.458
    30 |   2.6294 |  0.483
    35 |   2.5327 |  0.531
    40 |   2.4170 |  0.566
    45 |   2.3834 |  0.575
    50 |   2.3860 |  0.566
[PointNetTrainer] Entrenamiento completo. Mejor loss: 2.3834
PointNetModel entrenado.


In [10]:
pn_preds  = pn_trainer.predict(X_test)
pn_probas = pn_trainer.predict_proba(X_test)

pn_top1 = accuracy_score(y_test, pn_preds)
pn_top3 = top_k_accuracy_score(y_test, pn_probas, k=3, labels=list(range(NUM_CLASSES)))
pn_f1   = f1_score(y_test, pn_preds, average="macro",
                   labels=list(range(NUM_CLASSES)), zero_division=0)

print("=" * 40)
print("PointNetModel — Resultados Test")
print("=" * 40)
print(f"  Top-1 Accuracy : {pn_top1:.4f}")
print(f"  Top-3 Accuracy : {pn_top3:.4f}")
print(f"  Macro F1       : {pn_f1:.4f}")

PointNetModel — Resultados Test
  Top-1 Accuracy : 0.2936
  Top-3 Accuracy : 0.4740
  Macro F1       : 0.2351


In [11]:
# Predicción individual de ejemplo
result_pn = pn_trainer.predict_single(sample_seq, top_k=5)

print(f"Etiqueta real : {true_label}")
print(f"Predicción    : {IDX2GLOSA[result_pn['predicted_class']]}  "
      f"(conf={result_pn['confidence']:.3f})")
print("Top-5:")
for r in result_pn["top_k"]:
    print(f"  {IDX2GLOSA[r['class']]:<20} {r['prob']:.3f}")

Etiqueta real : 001
Predicción    : 084  (conf=0.214)
Top-5:
  084                  0.214
  002                  0.125
  073                  0.112
  001                  0.075
  113                  0.062


---
## 5. Tabla comparativa

In [12]:
import pandas as pd

rows = [
    {
        "Modelo":        "DistanceModel (kNN)",
        "Top-1 Acc":     dist_top1,
        "Top-3 Acc":     dist_top3,
        "Macro F1":      dist_f1,
        "Entrenamiento": "Ninguno",
    },
    {
        "Modelo":        "PointNetModel",
        "Top-1 Acc":     pn_top1,
        "Top-3 Acc":     pn_top3,
        "Macro F1":      pn_f1,
        "Entrenamiento": f"{POINTNET_EPOCHS} épocas",
    },
]

results_df = pd.DataFrame(rows).set_index("Modelo")
results_df[["Top-1 Acc", "Top-3 Acc", "Macro F1"]] = \
    results_df[["Top-1 Acc", "Top-3 Acc", "Macro F1"]].applymap(lambda x: f"{x:.4f}")

display(results_df)

,Top-1 Acc,Top-3 Acc,Macro F1,Entrenamiento
Modelo,,,,
DistanceModel (kNN),0.0367,0.0795,0.0264,Ninguno
PointNetModel,0.2936,0.4740,0.2351,50 épocas


---
## 6. Gráficas comparativas

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"figure.dpi": 120, "font.size": 11})

modelos  = ["kNN", "PointNet"]
top1_vals = [dist_top1, pn_top1]
top3_vals = [dist_top3, pn_top3]
f1_vals   = [dist_f1,   pn_f1]

if transformer_available:
    modelos.append("Transformer")
    top1_vals.append(tf_top1)
    top3_vals.append(tf_top3)
    f1_vals.append(tf_f1)

x     = range(len(modelos))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar([i - width for i in x], top1_vals, width, label="Top-1",  color="#4C72B0")
bars2 = ax.bar([i         for i in x], top3_vals, width, label="Top-3",  color="#55A868")
bars3 = ax.bar([i + width for i in x], f1_vals,   width, label="Macro F1", color="#C44E52")

ax.set_ylabel("Score")
ax.set_title("Comparación de modelos — Test set")
ax.set_xticks(list(x))
ax.set_xticklabels(modelos)
ax.set_ylim(0, 1.05)
ax.legend()
ax.yaxis.grid(True, alpha=0.3)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.2f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("comparacion_modelos.png", bbox_inches="tight")
plt.show()
print("Guardado: comparacion_modelos.png")

---
## 7. Matriz de confusión — Top 20 glosas más frecuentes

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(y_true, y_pred, idx2glosa, title, top_n=20):
    """
    Grafica la matriz de confusión para las top_n clases más frecuentes en test.
    """
    # Seleccionar top_n clases por frecuencia en y_true
    unique, counts = np.unique(y_true, return_counts=True)
    top_classes    = unique[np.argsort(counts)[::-1][:top_n]]

    # Filtrar muestras de esas clases
    mask    = np.isin(y_true, top_classes)
    y_t_filt = y_true[mask]
    y_p_filt = y_pred[mask]

    cm     = confusion_matrix(y_t_filt, y_p_filt, labels=top_classes)
    labels = [idx2glosa[c] for c in top_classes]

    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=labels, yticklabels=labels,
        linewidths=0.5, ax=ax,
    )
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    ax.set_title(f"{title} — Top {top_n} glosas")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    fname = title.replace(" ", "_").lower() + "_confusion.png"
    plt.savefig(fname, bbox_inches="tight")
    plt.show()
    print(f"Guardado: {fname}")


plot_confusion_matrix(y_test, dist_preds, IDX2GLOSA, "DistanceModel (kNN)")

In [ ]:
plot_confusion_matrix(y_test, pn_preds, IDX2GLOSA, "PointNetModel")

In [ ]:
if transformer_available:
    plot_confusion_matrix(tf_labels, tf_preds, IDX2GLOSA, "LSMTransformer")

---
## 8. Análisis de errores — glosas más confundidas

In [ ]:
def top_confusions(y_true, y_pred, idx2glosa, n=10):
    """
    Muestra los n pares (real, predicho) más frecuentes donde el modelo se equivoca.
    """
    errors = [(idx2glosa[t], idx2glosa[p])
              for t, p in zip(y_true, y_pred) if t != p]

    if not errors:
        print("Sin errores en el test set.")
        return

    from collections import Counter
    counts = Counter(errors).most_common(n)

    df_err = pd.DataFrame(counts, columns=["(Real, Predicho)", "Frecuencia"])
    display(df_err)


print("── DistanceModel ──")
top_confusions(y_test, dist_preds, IDX2GLOSA)

print("\n── PointNetModel ──")
top_confusions(y_test, pn_preds, IDX2GLOSA)

if transformer_available:
    print("\n── LSMTransformer ──")
    top_confusions(tf_labels, tf_preds, IDX2GLOSA)

---
## 9. Visualización de representaciones — UMAP

> Instala umap antes de correr: `pip install umap-learn`

In [ ]:
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    print("umap-learn no instalado. Saltando sección 9.")
    print("Instala con: pip install umap-learn")
    UMAP_AVAILABLE = False

In [ ]:
if UMAP_AVAILABLE and transformer_available:
    # Extraer representaciones del Transformer (pooled antes del head)
    tf_model.eval()
    reps, rep_labels = [], []

    test_loader_rep = DataLoader(test_ds, batch_size=64,
                                 shuffle=False, collate_fn=collate_fn)

    with torch.no_grad():
        for batch in test_loader_rep:
            kpts  = batch["keypoints"].to(DEVICE)
            mask  = batch["valid_mask"].to(DEVICE)

            # Forward hasta el pooling (antes del glosa_head)
            from model import LSMTransformer
            h = tf_model.embedding(kpts)
            h = tf_model.pos_enc(h)
            h = tf_model.transformer(h, src_key_padding_mask=~mask)
            # Masked mean pooling
            mask_f = mask.float().unsqueeze(-1)
            pooled = (h * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)

            reps.append(pooled.cpu().numpy())
            rep_labels.extend(batch["label"].tolist())

    reps       = np.concatenate(reps, axis=0)   # (N_test, d_model)
    rep_labels = np.array(rep_labels)

    print(f"Representaciones: {reps.shape}")
    print("Corriendo UMAP...")

    reducer   = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=15, min_dist=0.1)
    embedding = reducer.fit_transform(reps)     # (N_test, 2)

    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(
        embedding[:, 0], embedding[:, 1],
        c=rep_labels, cmap="tab20",
        alpha=0.6, s=15,
    )
    ax.set_title("UMAP — Representaciones LSMTransformer (test set)")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    plt.colorbar(scatter, ax=ax, label="Clase")
    plt.tight_layout()
    plt.savefig("umap_transformer.png", bbox_inches="tight")
    plt.show()
    print("Guardado: umap_transformer.png")

---
## 10. Resumen final

In [ ]:
print("\n" + "="*55)
print(f"{'Modelo':<25} {'Top-1':>7} {'Top-3':>7} {'F1':>7}")
print("-"*55)
print(f"{'DistanceModel (kNN)':<25} {dist_top1:>7.4f} {dist_top3:>7.4f} {dist_f1:>7.4f}")
print(f"{'PointNetModel':<25} {pn_top1:>7.4f} {pn_top3:>7.4f} {pn_f1:>7.4f}")
if transformer_available:
    print(f"{'LSMTransformer':<25} {tf_top1:>7.4f} {tf_top3:>7.4f} {tf_f1:>7.4f}")
print("="*55)

# Ganador por Top-1
scores = {
    "DistanceModel (kNN)": dist_top1,
    "PointNetModel":       pn_top1,
}
if transformer_available:
    scores["LSMTransformer"] = tf_top1

ganador = max(scores, key=scores.get)
print(f"\nMejor Top-1 Accuracy: {ganador} ({scores[ganador]:.4f})")